In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/sundharhug24ad-cloud/exp1ml/refs/heads/main/student_performance_updated_1000.csv")

print(df.shape)
df.head()

(1000, 12)


,StudentID,Name,Gender,AttendanceRate,StudyHoursPerWeek,PreviousGrade,ExtracurricularActivities,ParentalSupport,FinalGrade,Study Hours,Attendance (%),Online Classes Taken
0,1.0,John,Male,85.0,15.0,78.0,1.0,High,80.0,4.8,59.0,False
1,2.0,Sarah,Female,90.0,20.0,85.0,2.0,Medium,87.0,2.2,70.0,True
2,3.0,Alex,Male,78.0,10.0,65.0,0.0,Low,68.0,4.6,92.0,False
3,4.0,Michael,Male,92.0,25.0,90.0,3.0,High,92.0,2.9,96.0,False
4,5.0,Emma,Female,NaN,18.0,82.0,2.0,Medium,85.0,4.1,97.0,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StudentID                  960 non-null    float64
 1   Name                       966 non-null    object 
 2   Gender                     952 non-null    object 
 3   AttendanceRate             960 non-null    float64
 4   StudyHoursPerWeek          950 non-null    float64
 5   PreviousGrade              967 non-null    float64
 6   ExtracurricularActivities  957 non-null    float64
 7   ParentalSupport            978 non-null    object 
 8   FinalGrade                 960 non-null    float64
 9   Study Hours                976 non-null    float64
 10  Attendance (%)             959 non-null    float64
 11  Online Classes Taken       975 non-null    object 
dtypes: float64(8), object(4)
memory usage: 93.9+ KB


In [5]:
target = "FinalGrade"

features = [
    col for col in df.columns
    if col not in ["FinalGrade", "StudentID", "Name"]
]

X = df[features]
y = df[target]

print("Features:")
print(features)

print("\nTarget:")
print(target)

Features:
['Gender', 'AttendanceRate', 'StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'ParentalSupport', 'Study Hours', 'Attendance (%)', 'Online Classes Taken']

Target:
FinalGrade


In [6]:
mask = y.notna()

X = X.loc[mask]
y = y.loc[mask]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (960, 9)
y shape: (960,)


In [7]:
categorical_columns = [
    col for col in features
    if X[col].dtype == "object"
]

numerical_columns = [
    col for col in features
    if col not in categorical_columns
]

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['AttendanceRate', 'StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'Study Hours', 'Attendance (%)']

Categorical columns:
['Gender', 'ParentalSupport', 'Online Classes Taken']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (768, 9)
Testing data: (192, 9)


In [9]:
svm_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

In [10]:
svm_model = Pipeline([
    ("preprocessor", svm_preprocessor),
    (
        "model",
        SVR(
            kernel="rbf",
            C=100,
            gamma="scale",
            epsilon=0.1
        )
    )
])

In [11]:
svm_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
svm_predictions = svm_model.predict(X_test)

print(svm_predictions[:10])

[81.5238768  96.23942002 85.20703214 85.50395606 91.97895469 81.63949892
 88.97127524 91.42202645 73.3025678  92.44447419]


In [14]:
svm_mae = mean_absolute_error(y_test, svm_predictions)
svm_rmse = np.sqrt(mean_squared_error(y_test, svm_predictions))
svm_r2 = r2_score(y_test, svm_predictions)

print("SVM Results")
print("MAE :", svm_mae)
print("RMSE:", svm_rmse)
print("R2  :", svm_r2)

SVM Results
MAE : 9.098062072775095
RMSE: 11.411877588862117
R2  : -0.47769294151826447


In [15]:
rf_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numerical_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    )
])

In [16]:
rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [17]:
rf_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
rf_predictions = rf_model.predict(X_test)

print(rf_predictions[:10])

[75.66333333 77.62666667 80.39333333 79.65666667 79.1        79.51666667
 81.26333333 79.49333333 76.34       82.52666667]


In [19]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

Random Forest Results
MAE : 8.139114583333333
RMSE: 9.59437437948907
R2  : -0.04448765600932347


In [20]:
results = pd.DataFrame({
    "Model": ["SVM", "Random Forest"],
    "MAE": [svm_mae, rf_mae],
    "RMSE": [svm_rmse, rf_rmse],
    "R2": [svm_r2, rf_r2]
})

results

,Model,MAE,RMSE,R2
0,SVM,9.098062,11.411878,-0.477693
1,Random Forest,8.139115,9.594374,-0.044488


In [21]:
if rf_r2 > svm_r2:
    print("Random Forest performs better.")
else:
    print("SVM performs better.")

Random Forest performs better.
